# Modelamiento integral — Bank Marketing

## Despliegue de Soluciones Analíticas

**Objetivo:** comparar nueve configuraciones de clasificación, seleccionar un modelo orientado a priorización comercial, registrar los experimentos y artefactos en MLflow y guardar el modelo final.

### Antes de ejecutar
- Actualizar el repositorio con `git pull --ff-only origin main` cuando no haya un rebase pendiente.
- Recuperar los datos versionados con `dvc pull` únicamente si los archivos requeridos están gestionados por DVC.
- Activar el entorno `.venv-dvc` e instalar `pandas`, `numpy`, `scikit-learn`, `imbalanced-learn`, `xgboost`, `mlflow` y `joblib`.
- Iniciar el servidor MLflow configurado en `http://127.0.0.1:5000`.

### Criterios metodológicos
- Se reutilizan las particiones train/test guardadas en el repositorio; **no se recrea la partición original**.
- Se crea una partición interna de entrenamiento/validación a partir de `X_train` para los registros individuales de variantes.
- La comparación principal usa validación cruzada agrupada de cinco folds y predicciones out-of-fold (OOF).
- SMOTE y la ponderación se aplican exclusivamente al entrenamiento de cada fold.
- El test se evalúa al final. Este conjunto tuvo exposición histórica en evaluaciones anteriores del proyecto; por tanto, no se presenta como test completamente inédito.
- Las matrices cargadas ya están transformadas. Este notebook no reconstruye el preprocesamiento desde datos crudos; eso debe integrarse posteriormente en el artefacto de despliegue.


## 0. Importaciones y configuración general


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## 0. CONFIGURACIÓN GENERAL Y RUTAS


In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
TEST_SIZE_VALIDACION = 0.20
UMBRAL_REFERENCIA = 0.50
UMBRALES = np.round(np.arange(0.10, 1.00, 0.10), 2)
CAPACIDADES = (0.10, 0.20)

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENTO = "bank-marketing-modelamiento-integral"

def localizar_datos_procesados():
    """Busca los CSV de particiones existentes; no los vuelve a crear."""
    cwd = Path.cwd().resolve()
    candidatos = []

    for base in [cwd, *cwd.parents]:
        candidatos.extend([
            base / "data" / "processed",
            base / "notebooks" / "data" / "processed",
        ])

    requeridos = {
        "X_train_final.csv",
        "X_test_final.csv",
        "y_train.csv",
        "y_test.csv",
    }

    for carpeta in candidatos:
        if carpeta.exists() and requeridos.issubset(
            {p.name for p in carpeta.iterdir() if p.is_file()}
        ):
            return carpeta

    raise FileNotFoundError(
        "No se encontró una carpeta data/processed con los cuatro CSV "
        "de particiones. Actualiza el repositorio y verifica la ruta."
    )


RUTA_PROCESADOS = localizar_datos_procesados()
print("Archivos de particiones:", RUTA_PROCESADOS)

## 1. CARGAR PARTICIONES EXISTENTES DESDE EL REPOSITORIO


In [ ]:
# index_col=0 recupera el índice original exportado desde el EDA.
X_train = pd.read_csv(
    RUTA_PROCESADOS / "X_train_final.csv",
    index_col=0,
)
X_test = pd.read_csv(
    RUTA_PROCESADOS / "X_test_final.csv",
    index_col=0,
)
y_train = pd.read_csv(
    RUTA_PROCESADOS / "y_train.csv",
    index_col=0,
).squeeze("columns").astype(int)
y_test = pd.read_csv(
    RUTA_PROCESADOS / "y_test.csv",
    index_col=0,
).squeeze("columns").astype(int)

print("\nDIMENSIONES DE LAS PARTICIONES EXISTENTES")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

assert isinstance(y_train, pd.Series)
assert isinstance(y_test, pd.Series)
assert X_train.index.equals(y_train.index), "Índices train desalineados."
assert X_test.index.equals(y_test.index), "Índices test desalineados."
assert X_train.columns.equals(X_test.columns), "Columnas train/test distintas."
assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)
assert set(y_train.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

columnas_indice = [
    c for c in X_train.columns
    if str(c).lower().startswith("unnamed")
    or str(c).lower() == "index"
]
if columnas_indice:
    raise ValueError(
        f"X contiene columnas que parecen índices: {columnas_indice}. "
        "Revisa cómo se exportaron las matrices desde el EDA."
    )

if X_train.isna().any().any() or X_test.isna().any().any():
    raise ValueError("Hay valores ausentes en las matrices procesadas.")

print("\nDistribución de clases train:")
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("\nDistribución de clases test:")
print(y_test.value_counts().sort_index())
print("\nValidación de estructura superada.")

## 2. PARTICIÓN INTERNA TRAIN / VALIDACIÓN


In [ ]:
# No se recrea el train/test original. Se separa un 20% de X_train
# para seleccionar modelos y umbrales sin utilizar X_test.
indices_train_model, indices_val = train_test_split(
    X_train.index,
    test_size=TEST_SIZE_VALIDACION,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

X_train_model = X_train.loc[indices_train_model].copy()
X_val = X_train.loc[indices_val].copy()
y_train_model = y_train.loc[indices_train_model].copy()
y_val = y_train.loc[indices_val].copy()

print("\nPARTICIÓN INTERNA")
print("Entrenamiento interno:", X_train_model.shape)
print("Validación:", X_val.shape)
print("Positivos train interno:", f"{y_train_model.mean():.2%}")
print("Positivos validación:", f"{y_val.mean():.2%}")

assert X_train_model.index.equals(y_train_model.index)
assert X_val.index.equals(y_val.index)
assert set(X_train_model.index).isdisjoint(set(X_val.index))
assert set(X_train_model.index).isdisjoint(set(X_test.index))
assert set(X_val.index).isdisjoint(set(X_test.index))

## 3. FUNCIONES DE MÉTRICAS


In [ ]:
def calcular_metricas_ranking(y_real, probabilidades, k):
    y_array = np.asarray(y_real)
    proba = np.asarray(probabilidades)
    n_top = max(1, int(np.ceil(len(y_array) * k)))
    posiciones = np.argsort(-proba)[:n_top]
    precision_k = float(y_array[posiciones].mean())
    prevalencia = float(y_array.mean())
    lift_k = precision_k / prevalencia if prevalencia > 0 else np.nan
    return precision_k, float(lift_k), n_top


def calcular_metricas_completas(y_real, probabilidades, umbral=0.50):
    y_array = np.asarray(y_real).astype(int)
    proba = np.asarray(probabilidades)
    pred = (proba >= umbral).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array, pred, labels=[0, 1]
    ).ravel()

    p10, l10, n10 = calcular_metricas_ranking(y_array, proba, 0.10)
    p20, l20, n20 = calcular_metricas_ranking(y_array, proba, 0.20)

    return {
        "precision": float(precision_score(y_array, pred, zero_division=0)),
        "recall": float(recall_score(y_array, pred, zero_division=0)),
        "f1": float(f1_score(y_array, pred, zero_division=0)),
        "pr_auc": float(average_precision_score(y_array, proba)),
        "roc_auc": float(roc_auc_score(y_array, proba)),
        "precision_at_10": p10,
        "lift_at_10": l10,
        "precision_at_20": p20,
        "lift_at_20": l20,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "positivos_predichos": int(pred.sum()),
        "porcentaje_seleccionado": float(pred.mean()),
        "clientes_top_10": int(n10),
        "clientes_top_20": int(n20),
    }

## 4. DEFINIR LOS NUEVE MODELOS CANDIDATOS


In [ ]:
def crear_modelos(ratio_positivo=None):
    """Crea instancias nuevas para cada fold o entrenamiento final."""
    if ratio_positivo is None:
        ratio_positivo = (
            (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        )

    return {
        "reglog_sin_balanceo": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE,
            )),
        ]),

        "reglog_balanceada": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),

        "random_forest_sin_balanceo": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

        "random_forest_balanceado": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),

        "gradient_boosting": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "gradient_boosting_ponderado": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "gradient_boosting_smote": GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
        ),

        "xgboost_sin_balanceo": XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=1,
            n_jobs=-1,
        ),

        "xgboost_ponderado": XGBClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=RANDOM_STATE,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=ratio_positivo,
            n_jobs=-1,
        ),
    }


def ajustar_modelo(nombre, modelo, X, y):
    """Aplica balanceo exclusivamente sobre el fold de entrenamiento."""
    if nombre == "gradient_boosting_ponderado":
        pesos = compute_sample_weight(
            class_weight="balanced",
            y=np.asarray(y),
        )
        modelo.fit(X, y, sample_weight=pesos)

    elif nombre == "gradient_boosting_smote":
        smote = SMOTE(random_state=RANDOM_STATE)
        X_res, y_res = smote.fit_resample(X, y)
        modelo.fit(X_res, y_res)

    elif nombre == "xgboost_ponderado":
        conteos = pd.Series(y).value_counts()
        ratio = float(conteos.get(0, 0) / max(conteos.get(1, 1), 1))
        modelo.set_params(scale_pos_weight=ratio)
        modelo.fit(X, y)

    else:
        modelo.fit(X, y)

    return modelo


modelos_base = crear_modelos()
print("\nModelos candidatos:", list(modelos_base.keys()))

## 5. VALIDACIÓN CRUZADA AGRUPADA (5 FOLDS)


In [ ]:
# Agrupa filas con el mismo perfil de predictores para evitar que
# un perfil repetido aparezca simultáneamente en train y validación.
hash_train = pd.util.hash_pandas_object(X_train, index=False)
groups_train = pd.factorize(hash_train)[0]

cv_agrupada = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

resultados_folds = []
probabilidades_oof = {
    nombre: np.full(len(X_train), np.nan)
    for nombre in modelos_base
}

for fold, (idx_fit, idx_valid) in enumerate(
    cv_agrupada.split(X_train, y_train, groups_train),
    start=1,
):
    X_fold_train = X_train.iloc[idx_fit]
    X_fold_val = X_train.iloc[idx_valid]
    y_fold_train = y_train.iloc[idx_fit]
    y_fold_val = y_train.iloc[idx_valid]

    grupos_fit = set(groups_train[idx_fit])
    grupos_valid = set(groups_train[idx_valid])
    assert grupos_fit.isdisjoint(grupos_valid)

    print(f"\nFOLD {fold}/{N_SPLITS}")

    for nombre in modelos_base:
        modelo = crear_modelos()[nombre]
        modelo = ajustar_modelo(
            nombre, modelo, X_fold_train, y_fold_train
        )

        proba = modelo.predict_proba(X_fold_val)[:, 1]
        probabilidades_oof[nombre][idx_valid] = proba

        metricas = calcular_metricas_completas(
            y_fold_val, proba, UMBRAL_REFERENCIA
        )
        metricas.update({"modelo": nombre, "fold": fold})
        resultados_folds.append(metricas)

        print(
            f"{nombre}: PR-AUC={metricas['pr_auc']:.4f}; "
            f"Lift@10={metricas['lift_at_10']:.4f}"
        )

for nombre, proba in probabilidades_oof.items():
    if np.isnan(proba).any():
        raise AssertionError(f"Predicciones OOF incompletas: {nombre}")

resultados_cv_agrupada = pd.DataFrame(resultados_folds)

metricas_cv = [
    "precision", "recall", "f1", "pr_auc", "roc_auc",
    "precision_at_10", "lift_at_10",
    "precision_at_20", "lift_at_20",
]

resumen_cv = (
    resultados_cv_agrupada
    .groupby("modelo")[metricas_cv]
    .agg(["mean", "std"])
)

print("\nRESUMEN DE VALIDACIÓN CRUZADA AGRUPADA")
print(resumen_cv.round(4))

## 6. EVALUACIÓN DE UMBRALES CON PREDICCIONES OOF


In [ ]:
resultados_umbrales = []
y_train_array = y_train.to_numpy()

for nombre, proba in probabilidades_oof.items():
    for umbral in UMBRALES:
        pred = (proba >= umbral).astype(int)
        resultados_umbrales.append({
            "modelo": nombre,
            "umbral": float(umbral),
            "precision": float(
                precision_score(y_train_array, pred, zero_division=0)
            ),
            "recall": float(
                recall_score(y_train_array, pred, zero_division=0)
            ),
            "f1": float(
                f1_score(y_train_array, pred, zero_division=0)
            ),
            "positivos_predichos": int(pred.sum()),
            "porcentaje_seleccionado": float(pred.mean()),
        })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

mejores_umbrales_f1 = (
    tabla_umbrales
    .sort_values(["modelo", "f1"], ascending=[True, False])
    .groupby("modelo")
    .head(1)
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)

print("\nMEJOR UMBRAL POR MODELO SEGÚN F1 (OOF)")
print(mejores_umbrales_f1.round(4).to_string(index=False))

## 7A. Registro individual de las cuatro variantes de Juan en MLflow

Esta sección complementa la comparación OOF: entrena cada variante sobre `X_train_model`, la evalúa sobre `X_val` y registra **métricas, parámetros y artefacto del modelo** en el experimento `bank-marketing-boosting-balanceo`. El conjunto de validación no se utiliza para ajustar los modelos.


In [ ]:
# Variantes incorporadas por Juan: Gradient Boosting ponderado,
# Gradient Boosting con SMOTE, XGBoost base y XGBoost ponderado.
variantes_juan = [
    "gradient_boosting_ponderado",
    "gradient_boosting_smote",
    "xgboost_sin_balanceo",
    "xgboost_ponderado",
]

experimento_anterior = mlflow.get_experiment_by_name(
    "bank-marketing-boosting-balanceo"
)
if experimento_anterior is None:
    experimento_juan = mlflow.set_experiment(
        "bank-marketing-boosting-balanceo"
    )
else:
    experimento_juan = mlflow.set_experiment(
        "bank-marketing-boosting-balanceo"
    )

metricas_validacion_juan = {}
modelos_juan_entrenados = {}

for nombre in variantes_juan:
    modelo = crear_modelos()[nombre]
    modelo = ajustar_modelo(
        nombre,
        modelo,
        X_train_model,
        y_train_model,
    )

    probabilidades_val = modelo.predict_proba(X_val)[:, 1]
    metricas = calcular_metricas_completas(
        y_val,
        probabilidades_val,
        umbral=UMBRAL_REFERENCIA,
    )

    # No registrar conteos como métricas MLflow: se conservan en la tabla.
    metricas_mlflow = {
        clave: float(valor)
        for clave, valor in metricas.items()
        if isinstance(valor, (int, float, np.integer, np.floating))
        and np.isfinite(valor)
    }

    with mlflow.start_run(run_name=f"juan_{nombre}"):
        mlflow.log_params({
            "modelo": nombre,
            "evaluacion": "validacion_holdout_interna",
            "threshold": UMBRAL_REFERENCIA,
            "random_state": RANDOM_STATE,
            "n_train_interno": len(X_train_model),
            "n_validacion": len(X_val),
            "tecnica_balanceo": (
                "sample_weight" if nombre == "gradient_boosting_ponderado"
                else "SMOTE" if nombre == "gradient_boosting_smote"
                else "scale_pos_weight" if nombre == "xgboost_ponderado"
                else "ninguna"
            ),
        })
        mlflow.log_metrics(metricas_mlflow)

        if isinstance(modelo, XGBClassifier):
            mlflow.xgboost.log_model(
                xgb_model=modelo,
                name="modelo",
            )
        else:
            mlflow.sklearn.log_model(
                sk_model=modelo,
                name="modelo",
                skops_trusted_types=["sklearn.tree._tree.Tree"],
            )

    metricas_validacion_juan[nombre] = metricas
    modelos_juan_entrenados[nombre] = modelo

    print(
        f"Registrado en MLflow: {nombre} | "
        f"PR-AUC={metricas['pr_auc']:.4f} | "
        f"Lift@10={metricas['lift_at_10']:.4f}"
    )

tabla_validacion_juan = (
    pd.DataFrame.from_dict(metricas_validacion_juan, orient="index")
    .sort_values("lift_at_10", ascending=False)
)
print("\nMÉTRICAS DE VALIDACIÓN DE LAS CUATRO VARIANTES DE JUAN")
display(tabla_validacion_juan.round(4))

# Volver al experimento principal para el resto del flujo.
mlflow.set_experiment(EXPERIMENTO)

## 7. EVALUACIÓN POR CAPACIDAD COMERCIAL TOP 10% / TOP 20%


In [ ]:
resultados_capacidad = []

for nombre, proba in probabilidades_oof.items():
    for capacidad in CAPACIDADES:
        precision_k, lift_k, cantidad = calcular_metricas_ranking(
            y_train, proba, capacidad
        )
        n_top = cantidad
        posiciones = np.argsort(-proba)[:n_top]
        tp = int(y_train_array[posiciones].sum())

        resultados_capacidad.append({
            "modelo": nombre,
            "capacidad": capacidad,
            "clientes_contactados": n_top,
            "verdaderos_positivos": tp,
            "falsos_positivos": int(n_top - tp),
            "precision": precision_k,
            "recall_capturado": float(tp / y_train_array.sum()),
            "lift": lift_k,
        })

tabla_capacidad = pd.DataFrame(resultados_capacidad)

print("\nRESULTADOS OOF POR CAPACIDAD")
print(tabla_capacidad.round(4).to_string(index=False))


# Métricas Top-K calculadas por fold: más robustas ante diferencias
# de escala de probabilidades entre los modelos de cada fold.
resultados_capacidad_fold = []

for fold, (idx_fit, idx_valid) in enumerate(
    cv_agrupada.split(X_train, y_train, groups_train),
    start=1,
):
    y_fold = y_train.iloc[idx_valid].to_numpy()

    for nombre, proba_oof in probabilidades_oof.items():
        proba_fold = proba_oof[idx_valid]

        for capacidad in CAPACIDADES:
            n_top = max(1, int(np.ceil(len(y_fold) * capacidad)))
            posiciones = np.argsort(-proba_fold)[:n_top]
            y_top = y_fold[posiciones]
            tp = int(y_top.sum())
            precision_k = float(y_top.mean())
            lift_k = precision_k / y_fold.mean()

            resultados_capacidad_fold.append({
                "modelo": nombre,
                "fold": fold,
                "capacidad": capacidad,
                "precision": precision_k,
                "recall_capturado": float(tp / y_fold.sum()),
                "lift": float(lift_k),
                "clientes_contactados": n_top,
                "verdaderos_positivos": tp,
            })

tabla_capacidad_fold = pd.DataFrame(resultados_capacidad_fold)
resumen_capacidad_fold = (
    tabla_capacidad_fold
    .groupby(["modelo", "capacidad"])
    .agg({
        "precision": ["mean", "std"],
        "recall_capturado": ["mean", "std"],
        "lift": ["mean", "std"],
        "clientes_contactados": "mean",
        "verdaderos_positivos": "mean",
    })
)

print("\nESTABILIDAD TOP-K POR FOLD")
print(resumen_capacidad_fold.round(4))

## 8. COMPARACIÓN Y SELECCIÓN DEL CANDIDATO


In [ ]:
# La selección se basa en el promedio de Lift@10 por fold, no en un
# ranking global de probabilidades OOF que puede mezclar escalas entre folds.
lift10_por_modelo = (
    resumen_capacidad_fold
    .xs(0.10, level="capacidad")["lift"]
    [["mean", "std"]]
    .rename(columns={"mean": "lift10_mean", "std": "lift10_std"})
)

ranking_seleccion = (
    lift10_por_modelo
    .join(resumen_medias[["pr_auc", "roc_auc"]])
    .sort_values(
        ["lift10_mean", "pr_auc"],
        ascending=False
    )
)

print("\nRANKING DE SELECCIÓN: LIFT@10 MEDIO POR FOLD Y PR-AUC")
display(ranking_seleccion.round(4))

# Criterio explícito: mayor Lift@10 promedio entre folds;
# PR-AUC promedio se utiliza como desempate.
modelo_seleccionado = ranking_seleccion.index[0]

# Umbral candidato del modelo ganador: maximiza F1 en predicciones OOF.
fila_umbral = (
    tabla_umbrales[tabla_umbrales["modelo"] == modelo_seleccionado]
    .sort_values("f1", ascending=False)
    .iloc[0]
)
umbral_final = float(fila_umbral["umbral"])

print("\nMODELO CANDIDATO:", modelo_seleccionado)
print("Umbral candidato por F1 OOF:", umbral_final)
print(
    "Nota: Top 10% y Top 20% son reglas de capacidad comercial; "
    "no son equivalentes al umbral binario."
)

## 9. ENTRENAR EL MODELO SELECCIONADO CON TODO X_TRAIN


In [ ]:
modelo_final = crear_modelos()[modelo_seleccionado]
modelo_final = ajustar_modelo(
    modelo_seleccionado,
    modelo_final,
    X_train,
    y_train,
)

print("\nModelo final entrenado con todo X_train.")
print("Modelo:", modelo_seleccionado)
print("Registros:", len(X_train))
print("Variables:", X_train.shape[1])

## 10. EVALUACIÓN FINAL EN TEST (UNA SOLA VEZ)


In [ ]:
probabilidades_test = modelo_final.predict_proba(X_test)[:, 1]
metricas_test = calcular_metricas_completas(
    y_test,
    probabilidades_test,
    umbral=umbral_final,
)

print("\nEVALUACIÓN FINAL EN TEST")
print(pd.DataFrame([metricas_test]).round(4).to_string(index=False))

resultados_top_k_test = []
for capacidad in CAPACIDADES:
    precision_k, lift_k, cantidad = calcular_metricas_ranking(
        y_test, probabilidades_test, capacidad
    )
    posiciones = np.argsort(-probabilidades_test)[:cantidad]
    tp = int(y_test.to_numpy()[posiciones].sum())

    resultados_top_k_test.append({
        "capacidad": f"Top {int(capacidad * 100)}%",
        "clientes_contactados": cantidad,
        "verdaderos_positivos": tp,
        "falsos_positivos": cantidad - tp,
        "precision": precision_k,
        "recall_capturado": tp / y_test.sum(),
        "lift": lift_k,
    })

tabla_top_k_test = pd.DataFrame(resultados_top_k_test)
print("\nRESULTADOS TOP-K EN TEST")
print(tabla_top_k_test.round(4).to_string(index=False))

# Advertencia metodológica: el conjunto test ya tuvo exposición histórica
# en evaluaciones anteriores del proyecto. No describirlo como test virgen.

## 11. REGISTRO EN MLFLOW


In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENTO)

# Registrar resumen CV de cada candidato sin duplicar artefactos.
for nombre in resumen_medias.index:
    fila = resumen_medias.loc[nombre]

    with mlflow.start_run(run_name=f"cv_agrupada_{nombre}"):
        mlflow.log_params({
            "modelo": nombre,
            "validacion": "StratifiedGroupKFold",
            "n_splits": N_SPLITS,
            "random_state": RANDOM_STATE,
            "evaluacion": "OOF agrupada",
        })

        for metrica in metricas_cv:
            mlflow.log_metric(
                f"{metrica}_mean",
                float(fila[metrica]),
            )
            desviacion = (
                resultados_cv_agrupada
                .loc[resultados_cv_agrupada["modelo"] == nombre, metrica]
                .std()
            )
            mlflow.log_metric(
                f"{metrica}_std",
                float(desviacion) if pd.notna(desviacion) else 0.0,
            )

# Run del modelo final: parámetros, CV, umbral, test y artefacto.
with mlflow.start_run(run_name=f"modelo_final_{modelo_seleccionado}") as run:
    mlflow.log_params({
        "modelo_seleccionado": modelo_seleccionado,
        "umbral_final": umbral_final,
        "criterio_seleccion": "Lift@10 medio en validacion agrupada; desempate PR-AUC",
        "validacion": "StratifiedGroupKFold",
        "n_splits": N_SPLITS,
        "estrategia_comercial_principal": "Top 10%",
        "estrategia_comercial_alternativa": "Top 20%",
        "test_con_exposicion_historica": True,
    })

    for metrica in metricas_cv:
        mlflow.log_metric(
            f"cv_{metrica}",
            float(resumen_medias.loc[modelo_seleccionado, metrica]),
        )

    for metrica, valor in metricas_test.items():
        if isinstance(valor, (int, float, np.integer, np.floating)):
            mlflow.log_metric(f"test_{metrica}", float(valor))

    mlflow.set_tags({
        "tipo_modelo": "modelo_final_candidato",
        "validacion": "agrupada",
        "test": "exposicion_historica_documentada",
        "objetivo": "priorizacion_clientes_bank_marketing",
    })

    if isinstance(modelo_final, XGBClassifier):
        import mlflow.xgboost
        mlflow.xgboost.log_model(
            xgb_model=modelo_final,
            name="modelo_final",
        )
    else:
        mlflow.sklearn.log_model(
            sk_model=modelo_final,
            name="modelo_final",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

    run_id_final = run.info.run_id

print("\nMLflow run final:", run_id_final)

## 12. INTERPRETABILIDAD (SI EL MODELO TIENE FEATURE_IMPORTANCES_)


In [ ]:
if hasattr(modelo_final, "feature_importances_"):
    importancias = pd.Series(
        modelo_final.feature_importances_,
        index=X_train.columns,
        name="importancia",
    ).sort_values(ascending=False)

    print("\nTOP 15 VARIABLES IMPORTANTES")
    print(importancias.head(15).to_string())
else:
    importancias = None
    print(
        "\nEl modelo seleccionado no expone feature_importances_. "
        "No se calcula esta interpretación con este método."
    )

## 13. GUARDAR MODELO, MÉTRICAS Y METADATOS


In [ ]:
# Ubicar raíz del proyecto buscando .git.
def localizar_raiz_proyecto():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (base / ".git").exists():
            return base
    return Path.cwd().resolve()


RAIZ_PROYECTO = localizar_raiz_proyecto()
CARPETA_MODELOS = RAIZ_PROYECTO / "models"
CARPETA_REPORTES = RAIZ_PROYECTO / "reports" / "modeling"

CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)
CARPETA_REPORTES.mkdir(parents=True, exist_ok=True)

ruta_modelo = CARPETA_MODELOS / "modelo_final_seleccionado.joblib"
joblib.dump(modelo_final, ruta_modelo)

metadata = {
    "modelo_seleccionado": modelo_seleccionado,
    "umbral_final": umbral_final,
    "criterio_seleccion": "Lift@10 medio en validacion agrupada; desempate PR-AUC",
    "registros_train": int(len(X_train)),
    "registros_test": int(len(X_test)),
    "numero_variables": int(X_train.shape[1]),
    "validacion": "StratifiedGroupKFold",
    "n_splits": N_SPLITS,
    "mlflow_experiment": EXPERIMENTO,
    "mlflow_run_id": run_id_final,
    "nota_test": (
        "El conjunto test tuvo exposicion historica en evaluaciones previas; "
        "no se considera completamente inedito."
    ),
    "nota_despliegue": (
        "El modelo se entrenó con matrices ya transformadas. "
        "Para inferencia desde datos crudos se debe integrar el preprocesamiento "
        "del EDA y respetar exactamente las 94/95 columnas y su orden."
    ),
}

with open(
    CARPETA_MODELOS / "metadata_modelo_final.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

# Guardar tablas para el informe y la trazabilidad.
resultados_cv_agrupada.to_csv(
    CARPETA_REPORTES / "resultados_cv_agrupada.csv",
    index=False,
)
tabla_umbrales.to_csv(
    CARPETA_REPORTES / "evaluacion_umbrales_oof.csv",
    index=False,
)
tabla_capacidad.to_csv(
    CARPETA_REPORTES / "evaluacion_capacidad_oof.csv",
    index=False,
)
tabla_top_k_test.to_csv(
    CARPETA_REPORTES / "evaluacion_top_k_test.csv",
    index=False,
)
pd.DataFrame([metricas_test]).to_csv(
    CARPETA_REPORTES / "metricas_test_modelo_final.csv",
    index=False,
)
if importancias is not None:
    importancias.to_csv(
        CARPETA_REPORTES / "importancia_variables.csv",
        header=True,
    )

print("\nPROCESO FINALIZADO")
print("Modelo:", ruta_modelo)
print("Metadatos:", CARPETA_MODELOS / "metadata_modelo_final.json")
print("Reportes:", CARPETA_REPORTES)

## 14. Consideraciones para el despliegue y la documentación

1. El archivo `modelo_final_seleccionado.joblib` espera las mismas variables transformadas y en el mismo orden que `X_train`.
2. Para exponer predicciones desde la API se debe empaquetar también el preprocesamiento aplicado en el EDA, incluyendo imputación, codificación y generación de características.
3. La regla Top 10% o Top 20% representa capacidad de contacto; el umbral binario elegido por F1 es una decisión distinta.
4. Documentar en el informe que el conjunto de test tuvo exposición histórica en experimentos anteriores.
5. Los modelos de Juan quedan registrados individualmente como artefactos en `bank-marketing-boosting-balanceo`; el modelo seleccionado se registra en `bank-marketing-modelamiento-integral`.
